# Data Ingestions

In this notebook we will work with data ingestion of financial data and create a small datawarehouse. We will first ingest small data from growwapi and for now ingest data for One month i.e from `25th August 2025` till `25th August 2026`.
Since each sectors of the market has different amout of data we will split it into the top indexes.  They are:

1. Bank Nifty
2. Nifty IT
3. Nifty Phrama
4. Nifty Financial Services
5. Nifty FMCG
6. Nifty Infra
7. Nifty Oil and Gas
8. Nifty Media
9. Nifty Private bank
10. Nifty PSU bank
11. Nifty reality
12. Nifty Commodities
13. Nifty Health Care
14. Nifty  Consumer durables



In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from growwapi import GrowwAPI
import os
from dotenv import load_dotenv
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
import time
load = load_dotenv()

In [2]:
# Define explicit configuration properties

session = snowflake.connector.connect(
    account=os.getenv('account'),  # Do not include '.snowflakecomputing.com'
    user=os.getenv('user'),
    password=os.getenv('password'),
    warehouse=os.getenv('warehouse'),
    database=os.getenv('database'),
    schema=os.getenv('schema'),
    role=os.getenv('role')
)
print(session)

In [4]:
# Accessing API
GrowApi = os.getenv('GrowwAPi')
GrowScret = os.getenv('GrowwScret')

groww = GrowwAPI(GrowApi)

instrument_df = groww.get_all_instruments()


Ready to Groww!


## 1. Finance WH Design


In [5]:
success, nchunks, nrows, _ = write_pandas(
    conn=session,
    df=instrument_df,
    table_name="NiftyInstrument",
    auto_create_table=True,  # Generates the table layout
    overwrite=True           # Overwrites existing table
)


In [6]:
Equity_df = instrument_df[instrument_df['instrument_type'] == 'EQ']

success, nchunks, nrows, _ = write_pandas(
    conn=session,
    df=Equity_df,
    table_name="EquityInstruments",
    auto_create_table=True,  # Generates the table layout
    overwrite=True           # Overwrites existing table
)

/var/folders/zb/bl3vjblx59ggp4ygks32q_7r0000gn/T/ipykernel_25264/2064857547.py:3: UserWarning: Pandas Dataframe has non-standard index of type <class 'pandas.core.indexes.base.Index'> which will not be written. Consider changing the index to pd.RangeIndex(start=0,...,step=1) or call reset_index() to keep index as column(s)
  success, nchunks, nrows, _ = write_pandas(


In [7]:
bank_nifty = [
    "HDFCBANK", "ICICIBANK", "SBIN", "AXISBANK", "KOTAKBANK", 
    "INDUSINDBK", "BANKBARODA", "PNB", "AUBANK", "FEDERALBNK", 
    "IDFCFIRSTB", "SBICARD"
]
BankNifty = instrument_df[instrument_df['trading_symbol'].isin(bank_nifty)]
BankNifty.head(10)

,exchange,exchange_token,trading_symbol,groww_symbol,name,instrument_type,segment,series,isin,underlying_symbol,...,expiry_date,strike_price,lot_size,tick_size,freeze_quantity,is_reserved,buy_allowed,sell_allowed,internal_trading_symbol,is_intraday
80278,NSE,17971,SBICARD,NSE-SBICARD,SBI Cards & Payment,EQ,CASH,EQ,INE018E01016,NaN,...,NaN,NaN,1,0.05,NaN,NaN,1,1,SBICARD-EQ,1
80436,NSE,4668,BANKBARODA,NSE-BANKBARODA,Bank of Baroda,EQ,CASH,EQ,INE028A01039,NaN,...,NaN,NaN,1,0.01,NaN,NaN,1,1,BANKBARODA-EQ,1
80496,NSE,1333,HDFCBANK,NSE-HDFCBANK,HDFC Bank,EQ,CASH,EQ,INE040A01034,NaN,...,NaN,NaN,1,0.05,NaN,NaN,1,1,HDFCBANK-EQ,1
80573,NSE,3045,SBIN,NSE-SBIN,State Bank Of India,EQ,CASH,EQ,INE062A01020,NaN,...,NaN,NaN,1,0.1,NaN,NaN,1,1,SBIN-EQ,1
80738,NSE,5258,INDUSINDBK,NSE-INDUSINDBK,IndusInd Bank,EQ,CASH,EQ,INE095A01012,NaN,...,NaN,NaN,1,0.1,NaN,NaN,1,1,INDUSINDBK-EQ,1
80769,NSE,4963,ICICIBANK,NSE-ICICIBANK,ICICI Bank,EQ,CASH,EQ,INE090A01021,NaN,...,NaN,NaN,1,0.1,NaN,NaN,1,1,ICICIBANK-EQ,1
80801,NSE,11184,IDFCFIRSTB,NSE-IDFCFIRSTB,IDFC First Bank,EQ,CASH,EQ,INE092T01019,NaN,...,NaN,NaN,1,0.01,NaN,NaN,1,1,IDFCFIRSTB-EQ,1
81838,NSE,1023,FEDERALBNK,NSE-FEDERALBNK,The Federal Bank,EQ,CASH,EQ,INE171A01029,NaN,...,NaN,NaN,1,0.05,NaN,NaN,1,1,FEDERALBNK-EQ,1
81863,NSE,10666,PNB,NSE-PNB,PNB,EQ,CASH,EQ,INE160A01022,NaN,...,NaN,NaN,1,0.01,NaN,NaN,1,1,PNB-EQ,1
82148,NSE,5900,AXISBANK,NSE-AXISBANK,Axis Bank,EQ,CASH,EQ,INE238A01034,NaN,...,NaN,NaN,1,0.1,NaN,NaN,1,1,AXISBANK-EQ,1


In [8]:
bank_nifty = [
    "HDFCBANK", "ICICIBANK", "SBIN", "AXISBANK", "KOTAKBANK", 
    "INDUSINDBK", "BANKBARODA", "PNB", "AUBANK", "FEDERALBNK", 
    "IDFCFIRSTB", "SBICARD"
]

BankNifty = instrument_df[instrument_df['trading_symbol'].isin(bank_nifty)]
success, nchunks, nrows, _ = write_pandas(
    conn=session,
    df=BankNifty,
    table_name="Bank_Fact",
    auto_create_table=True,  # Generates the table layout
    overwrite=True           # Overwrites existing table
)

/var/folders/zb/bl3vjblx59ggp4ygks32q_7r0000gn/T/ipykernel_25264/1159348254.py:8: UserWarning: Pandas Dataframe has non-standard index of type <class 'pandas.core.indexes.base.Index'> which will not be written. Consider changing the index to pd.RangeIndex(start=0,...,step=1) or call reset_index() to keep index as column(s)
  success, nchunks, nrows, _ = write_pandas(


In [9]:
nifty_financial_services = [
    "HDFCBANK", "ICICIBANK", "AXISBANK", "KOTAKBANK", "SBIN", 
    "BAJFINANCE", "BAJAJFINSV", "SBILIFE", "HDFCLIFE", "CHOLAFIN"
]

FinancialServicesDf = instrument_df[instrument_df['trading_symbol'].isin(nifty_financial_services)]
success, nchunks, nrows, _ = write_pandas(
    conn=session,
    df=FinancialServicesDf,
    table_name="Financial_Services_Fact",
    auto_create_table=True,  # Generates the table layout
    overwrite=True           # Overwrites existing table
)

/var/folders/zb/bl3vjblx59ggp4ygks32q_7r0000gn/T/ipykernel_25264/1887651249.py:7: UserWarning: Pandas Dataframe has non-standard index of type <class 'pandas.core.indexes.base.Index'> which will not be written. Consider changing the index to pd.RangeIndex(start=0,...,step=1) or call reset_index() to keep index as column(s)
  success, nchunks, nrows, _ = write_pandas(


In [10]:
nifty_private_bank = [
    "HDFCBANK", "ICICIBANK", "AXISBANK", "KOTAKBANK", 
    "INDUSINDBK", "AUBANK", "FEDERALBNK", "IDFCFIRSTB", "RBLBANK", "CUB"
]

PrivateBankdf = instrument_df[instrument_df['trading_symbol'].isin(nifty_private_bank)]
success, nchunks, nrows, _ = write_pandas(
    conn=session,
    df=PrivateBankdf,
    table_name="Private_Bank_Fact",
    auto_create_table=True,  # Generates the table layout
    overwrite=True           # Overwrites existing table
)

/var/folders/zb/bl3vjblx59ggp4ygks32q_7r0000gn/T/ipykernel_25264/4154487461.py:7: UserWarning: Pandas Dataframe has non-standard index of type <class 'pandas.core.indexes.base.Index'> which will not be written. Consider changing the index to pd.RangeIndex(start=0,...,step=1) or call reset_index() to keep index as column(s)
  success, nchunks, nrows, _ = write_pandas(


In [11]:
nifty_psu_bank = [
    "SBIN", "BANKBARODA", "PNB", "CANBK", "UNIONBANK", 
    "IOB", "BANKINDIA", "MAHABANK", "CENTRALBK", "UCOBANK"
]

PSUBank = instrument_df[instrument_df['trading_symbol'].isin(nifty_psu_bank)]
success, nchunks, nrows, _ = write_pandas(
    conn=session,
    df=PSUBank,
    table_name="PSU_Bank_Fact",
    auto_create_table=True,  # Generates the table layout
    overwrite=True           # Overwrites existing table
)

/var/folders/zb/bl3vjblx59ggp4ygks32q_7r0000gn/T/ipykernel_25264/1345295665.py:7: UserWarning: Pandas Dataframe has non-standard index of type <class 'pandas.core.indexes.base.Index'> which will not be written. Consider changing the index to pd.RangeIndex(start=0,...,step=1) or call reset_index() to keep index as column(s)
  success, nchunks, nrows, _ = write_pandas(


In [23]:
start_time = '2026-08-01  00:00:00'
end_time = '2026-08-31  23:59:59'
all_records = []


for  symbol in BankNifty['trading_symbol']:
    try:
        print(f"Fetching data for :{symbol}")
        historical_data_reponse = groww.get_historical_candle_data(
        trading_symbol=symbol,
         exchange=groww.EXCHANGE_NSE,
        segment=groww.SEGMENT_CASH,
        start_time=start_time,
        end_time=end_time,
        interval_in_minutes=1 # 
        )
        if historical_data_reponse and 'candles' in historical_data_reponse:
            for candle in historical_data_reponse['candles']:
                record = {
                    'Symbol':symbol,
                    'Timestamp': candle[0],
                    'Open':candle[1],
                    ''
                    'High':candle[2],
                    'Low':candle[3],
                    'Close':candle[4],
                    'Volume':candle[5]
                }
                all_records.append(record)
        time.sleep(1)
    except Exception as e:
        print(f"Error fetching data for {symbol}: {e}")
        continue
final_df = pd.DataFrame(all_records)
final_df.head(10)

Fetching data for :SBICARD


/var/folders/zb/bl3vjblx59ggp4ygks32q_7r0000gn/T/ipykernel_25264/3392907396.py:9: DeprecationWarning: `get_historical_candle_data` is deprecated and will be removed in future releases. Please use `get_historical_candles` method instead. See https://groww.in/trade-api/docs/python-sdk/backtesting#get-historical-candle-data for more details.
  historical_data_reponse = groww.get_historical_candle_data(


Error fetching data for SBICARD: Access forbidden for this request.
Fetching data for :BANKBARODA
Error fetching data for BANKBARODA: Access forbidden for this request.
Fetching data for :HDFCBANK
Error fetching data for HDFCBANK: Access forbidden for this request.
Fetching data for :SBIN
Error fetching data for SBIN: Access forbidden for this request.
Fetching data for :INDUSINDBK
Error fetching data for INDUSINDBK: Access forbidden for this request.
Fetching data for :ICICIBANK
Error fetching data for ICICIBANK: Access forbidden for this request.
Fetching data for :IDFCFIRSTB
Error fetching data for IDFCFIRSTB: Access forbidden for this request.
Fetching data for :FEDERALBNK
Error fetching data for FEDERALBNK: Access forbidden for this request.
Fetching data for :PNB
Error fetching data for PNB: Access forbidden for this request.
Fetching data for :AXISBANK
Error fetching data for AXISBANK: Access forbidden for this request.
Fetching data for :KOTAKBANK
Error fetching data for KOTAKB

""


In [21]:
final_df


""


In [ ]:
session.close()